# ATLAS *Z*+jets Omnifold: Basic usage and setup

This notebook serves as an introduction to interacting with the ATLAS full-phase space measurement of $Z$+jets production in $pp$ collisions at $\sqrt{s}=13$ TeV.
It is based on the notebooks written for the previous [Multifold $Z$+jets measurement][https://gitlab.cern.ch/atlas-physics/public/sm-z-jets-omnifold-2024].
While the previous measurement targeted 24 observables, the full-phase space measurement is differential in the kinematics of every charged particle, making it a variable dimensional measurement.
For this reason the previously used HDF5 files are replaced with ROOT files and read using an uproot/awkward array interface.
Beyond this the introductory methods shown here are very similar.
Like in the previous round, two other notebooks provide more advanced instructions on how to access and create plots of the differential cross sections, how to obtain the associated covariance, and how to perform statistical compatibility tests: 
- One with results derived from pseudo-data, with a known target: [2_pseudo_results.ipynb](./2_pseudo_results.ipynb). 
- One with the actual measurements based on real data: [3_results.ipynb](./3_results.ipynb).
Finally three notebooks demonstrate advanced use cases of the un-binned spectra like those shown in the publication.

In [1]:
# Imports
import os
import numpy as np
import uproot
import pandas as pd
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

## Data and weights

The following data files are needed to construct the measurement.
All are provided in the ROOT file format:

- Default Monte-Carlo (MadGraph) sample: `ZjetOmnifold_5Jul2025_MGPy8FxFxPlusNonStrong_syst_Test_withdd.root`
- Alternative Monte-Carlo (Sherpa) sample: `ZjetOmnifold_Mar10_Sherpa2211_LookLike_MgFxFx_Test_V5.root`
- Truth Pseudodata, the target of the pseudo-measurement: `TruthPseudodata_Sherpa2211DY_Dibo_EW_PowhegPythiaTop_PosWeights_WithTracks_June2025_shuffled.root`
- Truth Sherpa sample: a set of files describing the truth-level Sherpa predictions

### Data download

To download all of the ROOT files above, run the cell below.
Note this will take a few GB of disk space (**TODO** add exact number)!

In [ ]:
### Download data (reconfigure once data are public)
import os
if os.path.isfile("files/data/multifold.h5"):
    print("Skipping download -- data already exists.")
else:
    !rm -f files.zip
    !wget https://zenodo.org/records/11507450/files/files.zip
    !unzip -o files.zip

try:
    from multifold_util import *
    import numpy as np
    import pandas as pd
    pd.options.mode.chained_assignment = None
    import matplotlib.pyplot as plt
    from tqdm import tqdm
    import os
    import pickle

    if os.path.isfile("files/data/multifold.h5"):
        print("Done!")
    else:
        raise Exception("File does not exist.")

except Exception as e:
    print(f"There was an error: {e}")

### Load and examine data

Once the data are downloaded, run the cells below to load it into memory.
In this notebook, we will only look at the total cross section predicted by the data measurement.
For this, we will need the default MC and Sherpa MC files.

In [ ]:
### Default: load the full measurement
n_events = None

### If running on Binder, which has a memory constraint of 2 GB, uncomment the following line to cut the size of the datasets approximately in half. 
### The results will not be identical to the ones presented in our paper, but can be used for demo & educational purposes.
# n_events = 200_000

In [ ]:
data_dir = "/pscratch/sd/k/kgreif/zjets_plot_staging/"
f = uproot.open(os.path.join(data_dir, "ZjetOmnifold_5Jul2025_MGPy8FxFxPlusNonStrong_syst_Test_withdd.root"))
t = f["OmniTree"]
f_sherpa = uproot.open(os.path.join(data_dir, "ZjetOmnifold_Mar10_Sherpa2211_LookLike_MgFxFx_Test_V5.root"))
t_sherpa = f_sherpa["OmniTree"]

In [ ]:
print(f"Examining the branches of the default MC TTree with {t.num_entries} entries")
t.show(name_width=40)

There are many branches. The most important ones are as follows:

- `truth_pass190`: A boolean flag which designates that the event falls in the fiducial volume targeted by the measurement. (In the future should filter events)
- `truth_pT_tracks`: The $p_T$ of all tracks in the events. This is a variable length (jagged) array with different numbers of entries per event.
- `truth_eta_tracks`: The same as above but giving the $\eta$ coordinate
- `truth_phi_tracks`: The same as above but giving the $\phi$ coordinate
- `truth_pdgId_tracks`: An integer designating the PDG ID of the truth charged hadron
- `truth_pT_l1`: The $p_T$ of the leading muon
- `truth_pT_l2`: The $p_T$ of the sub-leading muon
- `truth_eta_l1`: The $\eta$ of the leading muon
- `truth_eta_l2`: The $\eta$ of the sub-leading muon
- `truth_phi_l1`: The $\phi$ of the leading muon
- `truth_phi_l2`: The $\phi$ of the sub-leading muon

These branches are the raw ingredients that can be used to calculate an infinite number of observables, all of which will be constrained by this measurement.
However in this notebook, we are only interested in the total cross section which is only a function of the weights:

### Load and examine event weights

Next we will load the event weights that produce the data measurement

In [3]:
weight_dir = "../weight_storage/zjets-v4"
of_pseudodata = pd.read_hdf(os.path.join(weight_dir, "pd-weights.h5"), key="weights", mode="r")
of_pseudodata_hv = pd.read_hdf(os.path.join(weight_dir, "pd-weights.h5"), key="hv_weights", mode="r")

In [7]:
of_pseudodata

,weights_nominal,weights_ensemble_0,weights_ensemble_1,weights_ensemble_2,weights_ensemble_3,weights_ensemble_4,weights_ensemble_5,weights_ensemble_6,weights_ensemble_7,weights_ensemble_8,...,weights_bootstrap_data_86,weights_bootstrap_data_87,weights_bootstrap_data_88,weights_bootstrap_data_89,weights_bootstrap_data_90,weights_bootstrap_data_91,weights_bootstrap_data_92,weights_bootstrap_data_93,target_dd,weight_mc
0,0.044341,0.056762,0.037897,0.038798,0.048447,0.043870,0.047447,0.035405,0.047906,0.048433,...,0.051326,0.041875,0.045595,0.043673,0.040649,0.041340,0.042377,0.041991,0.009435,0.011355
1,0.016487,0.017510,0.021068,0.012138,0.016708,0.013613,0.017815,0.012283,0.015292,0.017510,...,0.017154,0.019988,0.020971,0.012912,0.016732,0.023439,0.017429,0.021096,0.018636,0.013356
2,0.052124,0.048983,0.050069,0.054199,0.062180,0.050857,0.052436,0.048343,0.055696,0.054595,...,0.054337,0.043203,0.042147,0.047266,0.048601,0.042833,0.043606,0.042015,0.010785,0.013756
3,0.009146,0.007646,0.009730,0.009916,0.008168,0.009630,0.008933,0.010213,0.008609,0.009752,...,0.009361,0.008787,0.010595,0.010476,0.008616,0.009107,0.009913,0.009082,0.009744,0.012562
4,0.068637,0.050307,0.076978,0.069168,0.057254,0.070570,0.083893,0.083991,0.058303,0.058961,...,0.051171,0.079322,0.084372,0.086593,0.091805,0.074585,0.090792,0.083027,0.020508,0.010395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1191153,0.196031,0.195993,0.160465,0.181211,0.217806,0.194939,0.236814,0.192565,0.222341,0.185447,...,0.198823,0.191462,0.216809,0.197717,0.121616,0.167122,0.113555,0.202538,0.103385,0.171014
1191154,0.316114,0.269088,0.295397,0.356366,0.340453,0.315297,0.321222,0.339234,0.307390,0.303621,...,0.329638,0.327096,0.335497,0.355538,0.347244,0.321288,0.330729,0.339767,0.682130,1.396617
1191155,0.439650,0.438989,0.465370,0.499756,0.417455,0.441270,0.410134,0.432152,0.428375,0.457689,...,0.466577,0.362573,0.389329,0.467339,0.450582,0.361899,0.438722,0.396515,0.331778,0.839839
1191156,0.038915,0.051718,0.031679,0.029456,0.043549,0.034928,0.041845,0.032435,0.043319,0.043681,...,0.039387,0.042366,0.037435,0.039533,0.036870,0.033053,0.034551,0.035657,0.100270,0.176122


We can see that the this event dataset contains 418,014 events, and the dimuon $p_\text{T}$ of the first event is 479 GeV, while the leading and subleading muons have $p_\text{T}$ 288 GeV and 198 GeV, respectively. We can also note that there are 276 "columns", i.e. properties of each event in this dataset file. These include the 24 kinematic observables, and about 250 event weights. Let's investigate this further with the next code box.

In [5]:
def printEventFeatures(dataset,desc):
    columns = np.array(dataset.keys())
    NmcBS=NdataBS=Nsys=Nens=0
    for column in columns:
        if "bootstrap_mc" in column:
            NmcBS+=1
        elif "bootstrap_data" in column:
            NdataBS+=1
        elif "ensemble" in column:
            Nens+=1
        elif "weight" in column:
            Nsys+=1
    print("\nFor {}, there are {} event properties:".format(desc,len(columns)))
    print("  {} weights, {} NN ensemble weights, {} MC and {} data bootstrap weights".format(Nsys,Nens,NmcBS,NdataBS))

printEventFeatures(of_pseudodata,"the main dataset")
printEventFeatures(of_pseudodata_hv,"the alternative, Sherpa-derived measurement")


For the main dataset, there are 120 event properties:
  15 weights, 10 NN ensemble weights, 0 MC and 94 data bootstrap weights

For the alternative, Sherpa-derived measurement, there are 2 event properties:
  2 weights, 0 NN ensemble weights, 0 MC and 0 data bootstrap weights


From the numbers displayed above, we can see that the three datasets all contain the 24 measured observables. 

There are a series of event weights that provide the measurement: central value + systematic variations. There is also a weight for the MC prediction as further detailed below. 
* The bootstrap weights provide statistical uncertainties. The MC bootstrap weights encodes the statistical uncertainties associated with the MC used to model the detector response (analogous to MC statistical uncertainty on the migration matrix in traditional unfolding). This uncertainty comes from the auxiliary sample used to train the NN models (i.e. it comes from the "training set" used to perform the actual unfolding, while the measurement files released are the "evaluation set").
* The data bootstrap weights encode the statistical uncertainty of the actual data.
* The NN ensemble weights corresponds to the results provided by 100 different neural networks, and their spread is taken as an uncertainty due to the 'NN instability'
* The final statistical uncertainty is the uncertainty on the datasets themselves (i.e. on the evaluation set).


## Obtaining measured cross sections, uncertainties and MC predictions

As described in the paper, the measurement is performed in a fiducial region defined by two opposite charge, prompt muons (prompt meaning that they do not originate from a hadron decay) that each fulfill $p_\mathrm{T}>20$ GeV and $|\eta|<2.5$. The dimuon system is further required to fulfill $m_{\mu\mu} \in (81,101)\,\text{GeV}$ and $p_\text{T}^{\mu\mu}>200\,\text{GeV}$, i.e. be consistent with a boosted $Z$ boson decaying to muons. Summing up `weights_nominal` of all events in the sample will return the central value of the measured cross section of the full fiducial volume. One can further construct a subregion based on any combination of selection criteria using on the 24 event observables (with certain caveats as discussed below) and obtain associated measured (and predicted) cross sections from the sum of weights of the events that fall in this region.

The central value in a subregion $A$ is given by the sum of nominal weights, `weights_nominal`, in the code, which has units femtobarn:

$$
\hat{\sigma}_A = \sum_{i\in A} w^\text{nom}_i.
$$

A systematically varied cross section corresponding to a nuisance parameter $(k)$ can be obtained using any of the alternative such weights:

$$
\hat{\sigma}_A^{(k)} = \sum_{i\in A} w^{(k)}_i.
$$

From such a variation, the absolute uncertainty amplitude would be given by the differnece to the central value: $\Delta_{A}^{(k)} = \hat{\sigma}_{A}^{(k)} - \hat{\sigma}_{A}$, and these variations should be treated as uncorrelated between each other, and fully correlated between bins. As a consequence, the covariance between two regions $A$ and $B$ from a given nuisance parameter $k$ is evaluated as $V_{A,B}^{(k)}=\Delta_{A}^{(k)}\,\Delta_{B}^{(k)}$, and the covariance from several such NPs is given by the sum: $V_{A,B} = \sum_k V_{A,B}^{(k)}$.

Statistical uncertainties are evaluated using bootstrap weights. The statistical covariance between two regions $A$ and $B$ is

$$
V^\text{stat}_{A,B} = \frac{1}{N_\text{BS}}\sum_{b=1}^{N_\mathrm{BS}}(\hat{\sigma}^{(b)}_A-\hat{\sigma}^\text{nom}_A)(\hat{\sigma}^{(b)}_B-\hat{\sigma}^\text{nom}_B).
$$

The code blocks below provide examples of how to extract some of these results. For more detailed checks, such as building the full covariance matrices, see the other two notebooks.

In [ ]:
# Method to obtain measured and predicted cross section of an event sample corresponding
# to a (fiducial) kinematic subregion
def printCrossSection(dataset,desc,wnom_name="weights_nominal"):
    meas_xsec = np.sum(dataset[wnom_name])
    pred_xsec = np.sum(dataset.weight_mc)
    # sumw2 = np.sum(dataset[wnom_name]**2)
    # Neff = meas_xsec*meas_xsec/sumw2 => see nEff method
    print("\nFor {}, we have:".format(desc))
    print("  Measured x-sec:     {:.2f} fb".format(meas_xsec))
    print("  MC-predicted x-sec: {:.2f} fb".format(pred_xsec))
    print("  Effective statistics: {:.1f}".format(nEff(dataset[wnom_name])))

printCrossSection(multifold,"nominal, MG5-based result, full fiducial phase space")
printCrossSection(multifold_sherpa,"Sherpa-based result, full fiducial phase space")
printCrossSection(multifold_nonDY,"Non-DY, full fiducial phase space")

# Define a fiducial subset ("A" in equations above)
print("\n\n===== FIDUCIAL VOLUME pT(ll) > 500 GeV =====")

meas_pT500 = multifold[multifold.pT_ll > 500]
sherpa_pT500 = multifold_sherpa[multifold_sherpa.pT_ll > 500]
nonDY_pT500 = multifold_nonDY[multifold_nonDY.pT_ll > 500]

printCrossSection(meas_pT500,"nominal, MG5-based result, in region pT(ll) > 500 GeV")
printCrossSection(sherpa_pT500,"Sherpa-based result, in region pT(ll) > 500 GeV")
printCrossSection(nonDY_pT500,"Non-Drell-Yan result, in region pT(ll) > 500 GeV")


The code above shows how to extract the measured central value, and the MC prediction, and effective statistics.

Both the nominal and the Sherpa results are obtained using Drell-Yan MC only. Their MC prediction does hence not include all signal processes, as EW $Zjj$ (aka 'VBF') and $ZV\to Zjj$ is missing. These processes are included in the non-Drell-Yan file.

The effective number of events (effective statistics) is calculated using $N_\text{eff} = \left(\sum_i w_i \right)^{2}/\sum_i w^2_i$. We can note that although there are 418k events in the full sample, after considering their weights, we effectively have only 198k events.

### Uncertainties

The measurement has a total of 29 sources of uncertainty. 

* 22 of these are encoded as alternative weights, and are propagated as standard nuisance parameters.
* Two sources are taken as the difference between the nominal result and results obtained by different MC samples. These are provided as separate datasets (2-point systematics). These account for the so-called hidden-variable uncertainty. 
* One uncertainty accounts for the data-driven unfolding uncertainty.
* Three uncertainties account for stochastic effects and are stored as bootstrap variations: 
    * The data statistical uncertainty
    * The MC statistical uncertainty on the training set
    * The neural network stochastic variation
* Finally, there is a statistical uncertainty of the provided MC events of the dataset.

In [ ]:
# Lists of uncertainty variations
syst_nps = ["pileup","lumi","topBackground", # pileup modelling, luminosity, bkg subtraction (top). 3 NPs
            "trackEffMain","trackEffJet","trackFake","trackPtScale", # track systematics. 4 NPs
            "muEffReco","muEffIso","muEffTrack","muEffTrig","muCalID","muCalMS","muCalResBias","muCalScale", # muon eff+calib. 8 NPs
            "theoryPSjet","theoryPSsoft","theoryMPI","theoryPSscale","theoryAlphaS","theoryQCD","theoryPDF"] # theory. 7 NPs

mc_train_stat  = [col for col in multifold.keys() if col.startswith("weights_bootstrap_mc")]
data_stat      = [col for col in multifold.keys() if col.startswith("weights_bootstrap_data")]
NN_stability   = [col for col in multifold.keys() if col.startswith("weights_ensemble")]

def printXsecUnc(dataset,desc,sherpa,nonDY):
    meas_xsec  = np.sum(dataset["weights_nominal"])    # sum w
    Vstat_meas = np.sum(dataset["weights_nominal"]**2) # sum w^2, statisical variance

    Vmeas = 0 # total, fractional variance
    print("\n---------\n{}".format(desc))
    print("   Measured x-sec:   {:.2f} fb".format(meas_xsec))
    print("\n   Systematics:")
    for NP in syst_nps:
        syst_xsec = np.sum(dataset["weights_"+NP])
        unc = (syst_xsec/meas_xsec-1) # fractional uncertainty amplitude
        Vmeas += unc*unc
        print("  {:>15s}:    {:5.2f}%".format(NP,unc*100))
        
    # Measurments performed with alternative MC samples 
    meas_sherpa = np.sum(sherpa["weights_nominal"]) # 'hidden variable uncertainty'
    meas_nonDY  = np.sum(nonDY["weights_nominal"]) # EW Zjj + diboson modelling 
    meas_DD  = np.sum(dataset["weights_dd"]) # Data-driven unfolding uncertainty
    meas_DD_target  = np.sum(dataset["target_dd"])
    print("  {:>15s}:    {:5.2f}%".format("DD unfold. unc.",(meas_DD/meas_DD_target-1)*100))
    
    Vmeas += (meas_DD/meas_DD_target-1)**2
    Vmeas += (meas_sherpa/meas_xsec-1)**2
    Vmeas += (meas_nonDY/meas_xsec-1)**2
    print("  {:>15s}:    {:5.2f}% (HV unfold. unc.)".format("Sherpa meas",(meas_sherpa/meas_xsec-1)*100))
    print("  {:>15s}:    {:5.2f}%".format("Non-DY meas",(meas_nonDY/meas_xsec-1)*100))
    print("  {:>15s}:    {:5.2f}%".format("Total syst",Vmeas**0.5*100))

    print("\n   Stochastic uncertainties:")
    print("  {:>15s}:    {:5.2f}%".format("MC stat",Vstat_meas**0.5/meas_xsec*100))
    Vmeas += Vstat_meas/meas_xsec/meas_xsec
    
    unc = np.std([np.sum(dataset[col]) for col in mc_train_stat])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("MC train stat",unc*100))

    unc = np.std([np.sum(dataset[col]) for col in data_stat])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("Data stat",unc*100))

    unc = np.std([np.sum(dataset[col]) for col in NN_stability])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("NN stability",unc*100))
    print("\n  {:>15s}:    {:5.2f}%".format("Total uncert",Vmeas**0.5*100))

In [ ]:
printXsecUnc(multifold,"Full fiducial phase space, i.e. pT(ll) > 200 GeV",multifold_sherpa,multifold_nonDY)

In [ ]:
printXsecUnc(meas_pT500,"pT(ll) > 500 GeV",sherpa_pT500,nonDY_pT500)